In [1]:
import gradio as gr
from openai import OpenAI

c:\Users\apoor\OneDrive\Desktop\git\GENAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
openai =  OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

: 

: 

: 

: 

: 

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code block"
def call_model(prompt):
    messages  = [
    {"role":"system","content":system_message},
    {"role":"user","content": prompt}
]
    response = openai.chat.completions.create(model="llama3.1:latest", messages= messages)
    return response.choices[0].message.content

: 

: 

: 

In [ ]:
call_model("What is today's date?")

: 

: 

: 

In [ ]:
gr.Interface(fn=call_model, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fbd81cd07fe6a09755.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


: 

: 

: 

: 

: 

### Adding authentication

Gradio makes it very easy to have user ids and password

In [ ]:
gr.Interface(fn = call_model, inputs = "textbox", outputs = "textbox", flagging_mode = "never").launch(inbrowser=True, auth=("apoorva", "secrethehe"))

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


: 

: 

: 

: 

: 

### Adding few fields

In [ ]:
message_input = gr.Textbox(label="Your question", info="Enter your question here", lines = 7)
#message_output = gr.Textbox(label = "Response:", lines=4)
message_output = gr.Markdown(label = "Response:")

view = gr.Interface(
    fn = call_model,
    title = "Ask me anything",
    inputs = [message_input],
    outputs = [message_output],
    examples = ["What is capital of France?", "Tell me a joke"],
    flagging_mode = "never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


: 

: 

: 

: 

: 

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code block"
def stream_llama3(prompt):
    message = [
        {"role":"system", "content":system_message},
        {"role":"user","content":prompt}
    ]
    stream = openai.chat.completions.create(
        model= "llama3.1:latest",
        messages=message,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    
    

: 

: 

: 

: 

: 

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code block"
def stream_gemma3(prompt):
    message = [
        {"role":"system", "content":system_message},
        {"role":"user","content":prompt}
    ]
    stream = openai.chat.completions.create(
        model= "gemma3:4b",
        messages=message,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    
    

: 

: 

: 

: 

: 

In [ ]:
message_input = gr.Textbox(label="Your question", info="Enter your question here", lines = 7)
#message_output = gr.Textbox(label = "Response:", lines=4)
message_output = gr.Markdown(label = "Response:")

view = gr.Interface(
    fn = stream_llama3,
    title = "Ask me anything",
    inputs = [message_input],
    outputs = [message_output],
    examples = ["What is capital of France?", "Tell me a joke"],
    flagging_mode = "never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


: 

: 

: 

: 

: 

In [ ]:
def stream_model(prompt, model):
    if model.lower().startswith("llama"):
        result = stream_llama3(prompt)
    elif model.lower().startswith("gemma"):
        result = stream_gemma3(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

: 

: 

: 

: 

: 

In [ ]:
message_input = gr.Textbox(label="Your message", info="Enter a message for teh LLM", lines=7)
model_selector = gr.Dropdown(["LLAMA","GEMMA"], label = "Select model", value="LLAMA")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn = stream_model,
    title = "LLMs",
    inputs = [message_input, model_selector],
    outputs = [message_output],
    examples= [
        ["Tell me a joke"],
        ["What is capital of India?"]
    ],
    flagging_mode = "never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


: 

: 

: 

: 

: 

### Chatbot with history using gradio

In [ ]:
system_prompt = "You are a helpful assistant that responds in markdown without code block"
def chat(message, history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model = "llama3.1:latest", messages = messages)
    return response.choices[0].message.content

gr.ChatInterface(fn = chat, type = "messages").launch()

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

: 

In [ ]:
import gradio as gr
import openai

# Define your system prompt
system_prompt = "You are a helpful assistant."

def chat(message, history):
    # Convert history into OpenAI format
    messages = [{"role": "system", "content": system_prompt}]
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": message})

    # Call OpenAI API
    response = openai.ChatCompletion.create(
        model="llama3.1:latest",  # or "llama3.1:latest" if supported in your setup
        messages=messages
    )

    return response.choices[0].message["content"]

# Launch Gradio
gr.ChatInterface(fn=chat).launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


: 